In [0]:
INSERT OVERWRITE TABLE bootcamp_gabriel.silver.dim_zona
SELECT DISTINCT 
    md5(partido || region) as id_zona, 
    zona_original, 
    partido, 
    region
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga de Dim_Propiedad
INSERT OVERWRITE TABLE bootcamp_gabriel.silver.dim_propiedad
SELECT DISTINCT 
    id as id_propiedad,
    ambientes,
    cochera,
    antiguedad,
    url
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga de Dim_Tiempo (Extraemos partes de la fecha)
INSERT OVERWRITE TABLE bootcamp_gabriel.silver.dim_tiempo
SELECT DISTINCT 
    CAST(fecha_proceso AS DATE) as id_fecha,
    day(fecha_proceso) as dia,
    month(fecha_proceso) as mes,
    year(fecha_proceso) as anio,
    quarter(fecha_proceso) as trimestre,
    date_format(fecha_proceso, 'EEEE') as dia_semana
FROM bootcamp_gabriel.silver.properties_silver;

In [0]:
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.dim_zona
SELECT DISTINCT md5(partido || region), zona_original, partido, region
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga Dim_Propiedad
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.dim_propiedad
SELECT DISTINCT id, ambientes, cochera, antiguedad, url
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga Dim_Tiempo
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.dim_tiempo
SELECT DISTINCT 
    CAST(fecha_proceso AS DATE),
    day(fecha_proceso), month(fecha_proceso), year(fecha_proceso),
    quarter(fecha_proceso), date_format(fecha_proceso, 'EEEE')
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga Fact_Ventas
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.fact_ventas
SELECT 
    monotonically_increasing_id(),
    md5(partido || region),
    id,
    CAST(fecha_proceso AS DATE),
    precio,
    metros_totales,
    metros_cubiertos,
    CASE WHEN metros_totales > 0 THEN ROUND(precio / metros_totales, 2) ELSE NULL END
FROM bootcamp_gabriel.silver.properties_silver;

In [0]:
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.dim_zona
SELECT DISTINCT 
    md5(partido || region), 
    zona_original, 
    partido, 
    region
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga de la Dimensión de Propiedad
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.dim_propiedad
SELECT DISTINCT 
    id, 
    ambientes, 
    cochera, 
    antiguedad, 
    url
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga de la Dimensión de Tiempo
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.dim_tiempo
SELECT DISTINCT 
    CAST(fecha_proceso AS DATE),
    day(fecha_proceso), 
    month(fecha_proceso), 
    year(fecha_proceso),
    quarter(fecha_proceso), 
    date_format(fecha_proceso, 'EEEE')
FROM bootcamp_gabriel.silver.properties_silver;

-- Carga de la Fact Table (con la métrica del Punto 3)
INSERT OVERWRITE TABLE bootcamp_gabriel.gold.fact_ventas
SELECT 
    monotonically_increasing_id(),
    md5(partido || region),
    id,
    CAST(fecha_proceso AS DATE),
    precio,
    metros_totales,
    metros_cubiertos,
    CASE 
        WHEN metros_totales > 0 THEN ROUND(precio / metros_totales, 2) 
        ELSE NULL 
    END
FROM bootcamp_gabriel.silver.properties_silver;